# PocketCoder — Standard MBPP Stage Evaluation (syntax + score, one harness for every checkpoint)

The **single unified harness** for Table 8 v2. One protocol for all five stages:
MBPP test split (n=500) · instruction prompt (`### Problem / ### Solution`, one assert shown) ·
**greedy** decoding · **seed 42** · new-tokens-only decode · syntax = `ast.parse` on extracted code ·
score = **pass@1** (all asserts) + **avg test pass rate** · 95% Wilson CIs · sandboxed execution
(child stdout silenced) · per-problem JSON with protocol block.

Set `HF_REPO` and run top-to-bottom. One run per checkpoint:

| Checkpoint | HF_REPO |
|---|---|
| Pretrained (base) | `Ananda100/pocketcoder100M`  ← current setting |
| Distilled | `Ananda100/pocketcoder100M-distilled` |
| Base + SFT | `Ananda100/100m-sft-python` |
| Distilled + SFT (final) | `Ananda100/PocketCoder` (already run: 8.6% / 13.9% / 95.4%) |
| + DPO | `Ananda100/pocketcoder-dpo` |


In [1]:
!pip install -q -U datasets transformers huggingface_hub safetensors tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.2 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

# ======================= CONFIG =======================
HF_REPO        = "Ananda100/pocketcoder100M"   # <-- change per stage (see table above)
NUM_PROBLEMS   = None      # None = full 500; 20 = smoke test
MAX_NEW_TOKENS = 256
SEED           = 42
TIMEOUT_S      = 5
OUT_FILE       = f"mbpp_stage_eval_{HF_REPO.split('/')[-1]}.json"
# ======================================================
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("HF token (blank if public): ") or None
print("Evaluating:", HF_REPO, "->", OUT_FILE)

HF token (blank if public): ··········
Evaluating: Ananda100/pocketcoder100M -> mbpp_stage_eval_pocketcoder100M.json


## Model architecture (identical to training) + greedy generate

In [3]:
import torch, math
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

torch.manual_seed(SEED)

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.flash = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1); att = self.attn_dropout(att); y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32022
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
    def forward(self, idx):
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        return self.lm_head(self.transformer.ln_f(x)[:, [-1], :])
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits = self(idx_cond)[:, -1, :]
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)   # greedy, deterministic
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx_next.item() == eos_token_id:
                break
        return idx

## Load checkpoint + tokenizer

In [4]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer
import json as _json

device = "cuda" if torch.cuda.is_available() else "cpu"
config_path  = hf_hub_download(repo_id=HF_REPO, filename="config.json", token=HF_TOKEN)
weights_path = hf_hub_download(repo_id=HF_REPO, filename="model.safetensors", token=HF_TOKEN)
with open(config_path) as f:
    config = GPTConfig(**_json.load(f))
model = GPT(config)
model.load_state_dict(load_file(weights_path))
model.to(device); model.eval()   # dropout OFF

try:
    tokenizer = AutoTokenizer.from_pretrained(HF_REPO, token=HF_TOKEN)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
assert tokenizer.eos_token_id is not None
assert config.vocab_size == len(tokenizer), "vocab mismatch between checkpoint and tokenizer!"
print(f"Loaded {HF_REPO} — {sum(p.numel() for p in model.parameters())/1e6:.2f}M params, vocab {config.vocab_size}, on {device}")

config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  482MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.29M [00:00<?, ?B/s]

Loaded Ananda100/pocketcoder100M — 95.87M params, vocab 32022, on cuda


## MBPP test split + instruction prompt

In [5]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "full", split="test")
if NUM_PROBLEMS:
    mbpp = mbpp.select(range(min(NUM_PROBLEMS, len(mbpp))))
print(f"MBPP test problems: {len(mbpp)}")

def build_prompt(ex):
    first_test = ex["test_list"][0] if ex["test_list"] else ""
    problem = f"{ex['text']}\nYour code should satisfy this test:\n{first_test}"
    return f"### Problem\n{problem}\n\n### Solution\n```python\n"

print(build_prompt(mbpp[0]))

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 87.2kB            

full/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

full/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  116kB            

full/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

MBPP test problems: 500
### Problem
Write a python function to remove first and last occurrence of a given character from the string.
Your code should satisfy this test:
assert remove_Occ("hello","l") == "heo"

### Solution
```python



## Sandboxed test execution (child process, silenced, hard kill)

In [6]:
import multiprocessing as mp

def _worker(code_s, setup, tests, q):
    # Silence the child completely: generated code often contains print loops,
    # and with fork() its stdout would flood the notebook cell. Set inside the
    # child only, so the notebook's own prints are untouched.
    import os as _os, sys as _sys
    _devnull = open(_os.devnull, "w")
    _sys.stdout = _devnull
    _sys.stderr = _devnull
    ns = {}
    try:
        if setup: exec(setup, ns)
        exec(code_s, ns)
        passed = 0
        for t in tests:
            try: exec(t, ns); passed += 1
            except Exception: pass
        q.put((passed, len(tests)))
    except Exception:
        q.put((0, len(tests)))

def run_tests(code_s, setup, tests, timeout_s=TIMEOUT_S):
    ctx = mp.get_context("fork"); q = ctx.Queue()
    p = ctx.Process(target=_worker, args=(code_s, setup, tests, q), daemon=True)
    p.start(); p.join(timeout_s)
    if p.is_alive():
        p.terminate(); p.join(1)
        if p.is_alive(): p.kill(); p.join()
        return 0, len(tests)
    try: return q.get_nowait()
    except Exception: return 0, len(tests)

## Generate + evaluate (greedy, new-tokens-only decode, syntax + score)

In [7]:
import ast
from tqdm.auto import tqdm

results = []
for ex in tqdm(mbpp, desc=f"MBPP eval [{HF_REPO.split('/')[-1]}]"):
    prompt = build_prompt(ex)
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    if len(ids) >= config.block_size - 8:
        continue
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=tokenizer.eos_token_id)

    # NEW-TOKENS-ONLY decode — never string-slice the full decode
    completion = tokenizer.decode(out[0][len(ids):].tolist(), skip_special_tokens=True)
    code_str = completion.split("```")[0].strip() if "```" in completion else completion.strip()

    # syntax validity (empty extraction counts as INVALID — ast.parse("") would pass)
    if not code_str:
        s_ok = False
    else:
        try:
            ast.parse(code_str); s_ok = True
        except SyntaxError:
            s_ok = False

    passed, total = run_tests(code_str, ex.get("test_setup_code", "") or "", ex["test_list"])
    results.append({"task_id": ex["task_id"], "syntax_ok": s_ok, "empty": not code_str,
                    "passed": passed, "total": total,
                    "full_pass": passed == total and total > 0})

print(f"Done: {len(results)} problems evaluated")

MBPP eval [pocketcoder100M]:   0%|          | 0/500 [00:00<?, ?it/s]

Done: 499 problems evaluated


## Report — pass@1, avg test pass, syntax validity, 95% CIs — and save JSON

In [8]:
import json, math

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n; denom = 1 + z*z/n
    c = (p + z*z/(2*n))/denom
    h = (z/denom)*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (max(0.0, c-h), min(1.0, c+h))

n = len(results)
k_pass   = sum(r["full_pass"] for r in results)
k_syntax = sum(r["syntax_ok"] for r in results)
n_empty  = sum(r["empty"] for r in results)
pass_at_1 = k_pass/n
syntax_validity = k_syntax/n
avg_test_pass = sum(r["passed"]/r["total"] for r in results if r["total"] > 0) / n
ci_pass, ci_syntax = wilson_ci(k_pass, n), wilson_ci(k_syntax, n)

print(f"--- {HF_REPO} : MBPP standard harness ---")
print(f"n = {n} | empty extractions = {n_empty}")
print(f"pass@1:          {pass_at_1:.1%}  (95% CI {ci_pass[0]:.1%} – {ci_pass[1]:.1%})")
print(f"avg test pass:   {avg_test_pass:.1%}")
print(f"syntax validity: {syntax_validity:.1%}  (95% CI {ci_syntax[0]:.1%} – {ci_syntax[1]:.1%})")

with open(OUT_FILE, "w") as f:
    json.dump({
        "repo": HF_REPO, "split": "test", "num_problems": n, "empty_extractions": n_empty,
        "pass_at_1": pass_at_1, "pass_at_1_ci95": ci_pass,
        "avg_test_pass_rate": avg_test_pass,
        "syntax_validity": syntax_validity, "syntax_validity_ci95": ci_syntax,
        "protocol": {
            "decoding": "greedy", "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
            "decode": "new-tokens-only, skip_special_tokens=True, cut at first ```",
            "prompt": "zero-shot, MBPP text + first test assertion, '### Problem / ### Solution' template",
            "syntax": "ast.parse on extracted code; EMPTY extraction counts as invalid",
            "execution": f"forked child (stdout silenced), {TIMEOUT_S}s hard-kill, test_setup_code + full test_list",
            "dropout": "eval mode (off)",
        },
        "per_problem": results,
    }, f, indent=2)
print(f"Saved: {OUT_FILE}")

--- Ananda100/pocketcoder100M : MBPP standard harness ---
n = 499 | empty extractions = 0
pass@1:          0.0%  (95% CI 0.0% – 0.8%)
avg test pass:   0.1%
syntax validity: 64.1%  (95% CI 59.8% – 68.2%)
Saved: mbpp_stage_eval_pocketcoder100M.json


## Notes

- **This is the one ruler for Table 8 v2** — run once per checkpoint (change `HF_REPO`), one row per JSON.
- Empty extractions are counted as **invalid** syntax (and reported), closing the `ast.parse("")` loophole —
  so the pretrained row's syntax may come out slightly below the earlier 64.1% if some extractions were empty.
  Whatever this harness prints is the citable number; state the convention in the caption.
- Already-known rows for reference: final model 8.6% / 13.9% / 95.4%; pretrained (pre-convention) 0.0% / 0.1% / 64.1%.
- Caption line for the paper: "MBPP test (n=500), instruction prompt with one assertion shown, greedy decoding
  (seed 42); syntax validity = fraction of non-empty extractions accepted by ast.parse; 95% Wilson CIs."